# Drilling seq2seq — evaluation report

Reads a single evaluation run from `results/{model_folder}/eval_{timestamp}/` and
produces diagnostic plots + tables. **Does not touch the model bundle or parquets** — this
notebook is pure-pandas/matplotlib and is safe to run in Colab on the Drive-mirrored
`results/` folder alone.

Artifacts consumed:
- `summary.json` — top-level metrics
- `per_step_accuracy.csv` — tidy (head, mode, step, top1, top3)
- `per_well_accuracy.csv` — per-well top-1 per head + duration MAE
- `confusion_{head}.csv` — top confused pairs per head
- `predictions.csv` — per-(sequence, step) predictions; optional tuple columns under constraints
- `run_config.json` — CLI args + frozen model config

Sections:
1. Setup & load
2. Headline numbers
3. Per-step accuracy curves (TF vs AR, all heads)
4. TF→AR gap per head
5. Hierarchy validity + conditional accuracy
6. Tuple top-K hit rate (constrained runs only)
7. Per-well accuracy distribution
8. Confusion pairs — top mistakes per head
9. Predictions.csv — step-wise error breakdown
10. Duration regression (when present)

## 1. Setup & load

Set `EVAL_DIR` to the absolute path of one `eval_<timestamp>` folder. Two common setups:

- **Local**: leave `USE_COLAB=False` and edit `EVAL_DIR` to an absolute path on your machine.
- **Colab**: set `USE_COLAB=True` — the cell will mount Drive and resolve `EVAL_DIR` under
  `/content/drive/MyDrive/drilling-e2e/results/...`.

If `EVAL_DIR` is a model folder (not yet an `eval_*` child), the newest `eval_*` subfolder
is picked automatically.

In [ ]:
USE_COLAB = False                                          # flip to True when running in Colab
EVAL_DIR  = "../results/seq2seq_N25_K8_T4_lr0.0001_p30_embed_state"   # model folder OR eval folder
COLAB_REPO_ROOT = "/content/drive/MyDrive/drilling-e2e"     # only used when USE_COLAB=True

import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if USE_COLAB:
    from google.colab import drive                         # noqa: F401
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    # If the user passed a relative path, resolve it under the Drive repo root.
    eval_dir = Path(EVAL_DIR)
    if not eval_dir.is_absolute():
        eval_dir = Path(COLAB_REPO_ROOT) / eval_dir.relative_to(eval_dir.anchor or '')
else:
    eval_dir = Path(EVAL_DIR).resolve()

# If user pointed at a model folder, pick the newest eval_* child.
if eval_dir.exists() and not (eval_dir / "summary.json").exists():
    candidates = sorted(p for p in eval_dir.glob("eval_*") if (p / "summary.json").exists())
    if not candidates:
        raise FileNotFoundError(f"No eval_* subfolder with summary.json under {eval_dir}")
    eval_dir = candidates[-1]

print(f"Using eval folder: {eval_dir}")
assert (eval_dir / "summary.json").exists(), f"summary.json not found in {eval_dir}"

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
summary      = json.loads((eval_dir / "summary.json").read_text())
per_step     = pd.read_csv(eval_dir / "per_step_accuracy.csv")
per_well     = pd.read_csv(eval_dir / "per_well_accuracy.csv")
run_config   = json.loads((eval_dir / "run_config.json").read_text()) if (eval_dir / "run_config.json").exists() else {}
confusion    = {}
for head in ["phase", "phase_step", "major_ops_code", "operation"]:
    path = eval_dir / f"confusion_{head}.csv"
    if path.exists():
        confusion[head] = pd.read_csv(path)

predictions_path = eval_dir / "predictions.csv"
predictions = pd.read_csv(predictions_path) if predictions_path.exists() else None

HIERARCHY = ["phase", "phase_step", "major_ops_code", "operation"]
K_TOP = int(summary.get("top_k", 3))
TOPK_COL = f"top{K_TOP}"

print(f"Model:           {summary.get('model_folder')}")
print(f"Split:           {summary.get('split')}")
print(f"n_sequences:     {summary.get('n_sequences'):,}")
print(f"n_wells:         {summary.get('n_wells')}")
print(f"top_k:           {K_TOP}")
print(f"constraints on:  {summary.get('constraints_enabled', '— (pre-constraint-decoder eval)')}")
print(f"|L|:             {summary.get('n_legal_tuples')}")
print(f"predictions.csv: {'loaded (' + f'{len(predictions):,}' + ' rows)' if predictions is not None else 'missing'}")

## 2. Headline numbers

Two things worth knowing at a glance: where the **TF ceiling** lands (model's representational
upper bound), and how far **AR** (deployment) falls below that ceiling. A large TF→AR gap
signals exposure-bias / cascade issues — which is exactly what constrained decoding is built
to close.

In [ ]:
tf = summary.get("tf", {})
ar = summary.get("ar", {})

rows = []
for h in HIERARCHY:
    rows.append({
        "head":          h,
        "TF top-1":      tf.get(f"{h}_top1"),
        f"TF top-{K_TOP}": tf.get(f"{h}_top{K_TOP}"),
        "AR top-1":      ar.get(f"{h}_top1"),
        f"AR top-{K_TOP}": ar.get(f"{h}_top{K_TOP}"),
        "TF-AR (top-1)": (tf.get(f"{h}_top1") or 0) - (ar.get(f"{h}_top1") or 0),
    })
headline = pd.DataFrame(rows).set_index("head")
headline.style.format("{:.4f}") if hasattr(headline, "style") else headline

## 3. Per-step accuracy — TF vs AR, per head

TF lines should be near-flat (each step has ground-truth context). AR lines should *decay*
across the horizon — the further into the future, the more cascaded errors accumulate.
Steep AR decay on MOC / operation is the classic cascade signature.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey=False)
for ax, head in zip(axes.flat, HIERARCHY):
    sub = per_step[per_step["head"] == head]
    for mode, style in (("tf", "--"), ("ar", "-")):
        s = sub[sub["mode"] == mode]
        if s.empty:
            continue
        ax.plot(s["step"], s["top1"], style, marker="o", label=f"{mode.upper()} top-1")
        ax.plot(s["step"], s[TOPK_COL], style, marker="x", alpha=0.6, label=f"{mode.upper()} top-{K_TOP}")
    ax.set_title(head)
    ax.set_xlabel("future step")
    ax.set_ylabel("accuracy")
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

## 4. TF → AR gap per head

How much does deployment-time decoding cost you, per head? Big gaps on MOC / operation are
common; phase usually holds up well because there are few classes and the transitions are
dominated by long runs of the same phase.

In [ ]:
gap_df = headline[["TF top-1", "AR top-1"]].copy()
gap_df["gap"] = gap_df["TF top-1"] - gap_df["AR top-1"]

fig, ax = plt.subplots(figsize=(7, 3.5))
x = np.arange(len(gap_df))
width = 0.35
ax.bar(x - width/2, gap_df["TF top-1"], width, label="TF top-1")
ax.bar(x + width/2, gap_df["AR top-1"], width, label="AR top-1")
for xi, gi in zip(x, gap_df["gap"]):
    ax.annotate(f"Δ={gi:.2f}", (xi, max(gap_df["TF top-1"].iloc[xi], gap_df["AR top-1"].iloc[xi]) + 0.02),
                ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(gap_df.index, rotation=15)
ax.set_ylim(0, 1.1)
ax.set_ylabel("top-1 accuracy")
ax.set_title("TF ceiling vs. AR deployment — per head")
ax.legend()
plt.tight_layout(); plt.show()

## 5. Hierarchy validity + conditional accuracy

`hierarchy_validity_rate` = fraction of AR predictions that form a legal (phase, phase_step,
major_ops_code, operation) tuple against the master data taxonomy. Under constrained decoding,
this should be 1.0 by construction.

Conditional accuracies tell you **where in the chain the model breaks down**. If `moc | phase_step`
is the worst, the model gets the step right but stumbles on MOC — which is usually the real
bottleneck on drilling sequences.

In [ ]:
ar = summary.get("ar", {})
validity = ar.get("hierarchy_validity_rate")
cond = ar.get("conditional_acc", {})

print(f"hierarchy_validity_rate = {validity:.4f}" if validity is not None else "hierarchy_validity_rate: n/a")
print()
print("Conditional top-1 accuracies (child correct | parent correct):")
for k, v in cond.items():
    if v is None: continue
    print(f"  {k:32s} {v:.4f}")

# Visualize conditional chain.
if cond:
    labels = list(cond.keys())
    vals   = [cond[k] if cond[k] is not None else np.nan for k in labels]
    fig, ax = plt.subplots(figsize=(7, 2.8))
    ax.barh(labels, vals)
    for i, v in enumerate(vals):
        if not np.isnan(v):
            ax.text(v + 0.01, i, f"{v:.2f}", va="center", fontsize=9)
    ax.set_xlim(0, 1.1)
    ax.set_xlabel("P(child correct | parent correct)")
    ax.set_title("Hierarchy-conditional accuracy")
    ax.invert_yaxis()
    plt.tight_layout(); plt.show()

## 6. Tuple top-K hit rate — constrained runs only

Did the ground-truth (phase, phase_step, major_ops_code, operation) tuple appear in the
chosen top-K legal tuples at this step? Available only when the eval run had constraints
enabled (look for `tuple_topK_overall` in `summary.json::ar`).

In [ ]:
overall_key = f"tuple_top{K_TOP}_overall"
perstep_key = f"tuple_top{K_TOP}_per_step"
if overall_key in ar and perstep_key in ar:
    per_step_vals = ar[perstep_key]
    overall = ar[overall_key]
    print(f"Overall tuple top-{K_TOP} hit rate: {overall:.4f}")

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(range(1, len(per_step_vals) + 1), per_step_vals, marker="o")
    ax.axhline(overall, linestyle=":", color="gray", label=f"overall = {overall:.3f}")
    ax.set_xlabel("future step")
    ax.set_ylabel(f"tuple top-{K_TOP} hit rate")
    ax.set_ylim(0, 1.02)
    ax.set_title("Ground-truth tuple within top-K legal tuples, per step")
    ax.legend()
    plt.tight_layout(); plt.show()
else:
    print("No tuple-level metrics in this run — constraints likely were off.")

## 7. Per-well accuracy distribution

How consistent is the model across wells? A wide spread on a head = sensitive to well-level
domain shift; worth investigating which wells are outliers. `well_accuracy_std` in
`summary.json` summarizes this.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for ax, head in zip(axes.flat, HIERARCHY):
    col = f"{head}_top1"
    if col not in per_well.columns:
        ax.set_visible(False); continue
    data = per_well[col].dropna()
    ax.hist(data, bins=30)
    ax.axvline(data.mean(), linestyle="--", color="black", label=f"mean={data.mean():.2f}")
    ax.set_title(f"{head} (n_wells={len(data)})")
    ax.set_xlabel("per-well top-1")
    ax.set_ylabel("wells")
    ax.set_xlim(0, 1)
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("\nWorst 5 wells by operation top-1:")
print(per_well.nsmallest(5, "operation_top1")[["well_name", "n_sequences"] + [f"{h}_top1" for h in HIERARCHY]].to_string(index=False))

## 8. Confusion pairs — top mistakes per head

Which classes does the model systematically swap? `pct_of_true` = fraction of rows with that
true label that got this particular wrong prediction. A high value means a systematic bias;
a long low-value tail means the model is uncertain but not consistently wrong.

In [ ]:
def plot_confusion(df, head, top_n=15):
    if df is None or df.empty:
        print(f"{head}: no confusion data"); return
    d = df.head(top_n).copy()
    labels = [f"{r.true_label} → {r.pred_label}" for r in d.itertuples()]
    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.3 * len(d))))
    ax.barh(labels, d["pct_of_true"])
    for i, (c, p) in enumerate(zip(d["count"], d["pct_of_true"])):
        ax.text(p + 0.005, i, f"n={c} ({p:.1%})", va="center", fontsize=8)
    ax.set_xlabel("pct_of_true")
    ax.set_title(f"{head} — top {len(d)} confusion pairs (off-diagonal)")
    ax.invert_yaxis()
    plt.tight_layout(); plt.show()

for head in HIERARCHY:
    plot_confusion(confusion.get(head), head)

## 9. `predictions.csv` — step-wise error breakdown

Recompute per-step accuracy directly from `predictions.csv` — useful as a sanity check
against `per_step_accuracy.csv`, and as a starting point for ad-hoc slicing (e.g. restrict
to a subset of wells).

In [ ]:
if predictions is None:
    print("predictions.csv not written for this run — skip.")
else:
    d = predictions.copy()
    # Per-step, per-head top-1 match.
    rows = []
    for head in HIERARCHY:
        true_col = f"true_{head}"
        pred_col = f"pred_{head}_top1"
        if true_col not in d.columns or pred_col not in d.columns:
            continue
        ok = (d[true_col] == d[pred_col]).groupby(d["step"]).mean().rename("top1")
        rows.append(ok.rename_axis("step").reset_index().assign(head=head))
    step_acc = pd.concat(rows) if rows else pd.DataFrame()

    if not step_acc.empty:
        fig, ax = plt.subplots(figsize=(8, 3.5))
        for head, sub in step_acc.groupby("head"):
            ax.plot(sub["step"], sub["top1"], marker="o", label=head)
        ax.set_xlabel("future step"); ax.set_ylabel("AR top-1 (recomputed)")
        ax.set_ylim(0, 1.02); ax.legend()
        ax.set_title("AR top-1 per step — recomputed from predictions.csv")
        plt.tight_layout(); plt.show()

    # Tuple top-K hit by step, if present.
    if "tuple_in_topk" in d.columns:
        tuple_hit = d.groupby("step")["tuple_in_topk"].mean()
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(tuple_hit.index, tuple_hit.values, marker="o")
        ax.set_xlabel("future step"); ax.set_ylabel(f"tuple top-{K_TOP} hit rate")
        ax.set_ylim(0, 1.02)
        ax.set_title("Ground-truth tuple within top-K — recomputed from predictions.csv")
        plt.tight_layout(); plt.show()

## 10. Duration regression (when predict_duration is on)

Only runs if the run included the duration head. MAE is in hours; `mae_long_ops_hours` zooms
in on ops with true duration > 8h (where mistakes cost the most).

In [ ]:
dur_keys = ["mae_hours", "medae_hours", "p95_abs_err_hours", "mae_long_ops_hours", "r2_long_ops"]
dur_stats = {k: ar[k] for k in dur_keys if k in ar}
if not dur_stats:
    print("No duration metrics in this run.")
else:
    print("Duration regression:")
    for k, v in dur_stats.items():
        print(f"  {k:28s} {v:.3f}" if v is not None else f"  {k:28s} n/a")

    if predictions is not None and {"true_duration_hours", "pred_duration_hours"}.issubset(predictions.columns):
        d = predictions.dropna(subset=["true_duration_hours", "pred_duration_hours"]).copy()
        d["abs_err"] = (d["pred_duration_hours"] - d["true_duration_hours"]).abs()
        mae_per_step = d.groupby("step")["abs_err"].mean()

        fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
        axes[0].plot(mae_per_step.index, mae_per_step.values, marker="o")
        axes[0].set_xlabel("future step"); axes[0].set_ylabel("MAE hours")
        axes[0].set_title("Duration MAE per step")

        sample = d.sample(min(3000, len(d)), random_state=0)
        axes[1].scatter(sample["true_duration_hours"], sample["pred_duration_hours"], alpha=0.25, s=8)
        lims = [0, max(sample["true_duration_hours"].max(), sample["pred_duration_hours"].max(), 1)]
        axes[1].plot(lims, lims, linestyle=":", color="black")
        axes[1].set_xlim(lims); axes[1].set_ylim(lims)
        axes[1].set_xlabel("true hours"); axes[1].set_ylabel("predicted hours")
        axes[1].set_title("Pred vs true (random 3K sample)")
        plt.tight_layout(); plt.show()

## What to look at first

- **Constraints on vs. off ablation**: run the eval twice (once with `--no-constraints`), load
  each `eval_*` folder in this notebook, and diff the headline table + per-step AR curves.
  Under constraints you expect `hierarchy_validity_rate → 1.0`; MOC / operation top-1 AR to
  improve; TF numbers unchanged.
- **Steep AR decay** on a head = cascade. Constraints help; sequence-level fine-tuning and
  head coupling help more.
- **Low `moc | phase_step` conditional** = model gets the step right, fumbles the MOC. This
  is usually the real bottleneck in drilling sequences.
- **Skewed per-well histogram** = handful of wells dragging the average down. Check the
  worst-5 table and inspect their Operator / Rig_Contractor in the raw data — could indicate
  an under-represented drilling program.